# Customer Churn Dataset: Data Quality and Exploratory Analysis
**Track:** Data Analytics
**Level:** Beginner-friendly

## Objective
Audit and clean a customer churn dataset, then document the quality decisions and initial patterns.

### Checklist
- [x] Profile columns, data types, missing values, outliers, and duplicates.
- [x] Clean the data with documented assumptions and validation checks.
- [x] Calculate summary statistics and inspect meaningful distributions.
- [x] Export the clean dataset and a data dictionary.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")


ModuleNotFoundError: No module named 'pandas'

: 

## 1. Load Data
First, we load the dataset and take a quick look at the first few rows.


In [ ]:
# Load dataset
df = pd.read_csv('customer_churn_sample.csv')
display(df.head())


## 2. Data Profiling
Let's profile the columns, data types, missing values, outliers, and duplicates.


In [ ]:
# Check data types and missing values
print("--- Data Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicates ---")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")


### Summary Statistics
Let's look at the summary statistics for numerical and categorical columns.


In [ ]:
# Numerical summary
display(df.describe())

# Categorical summary
display(df.describe(include=['O']))


## 3. Data Cleaning
Based on the profiling:
1. **Handling Missing Values:** We will check if there are any `NaN` values and decide on imputation (e.g., median for numeric, mode for categorical).
2. **Handling Duplicates:** We will drop exact duplicate rows to prevent bias.
3. **Data Type Casting:** We'll ensure `TotalCharges` is properly cast to numeric, handling any potential string errors (like empty strings or spaces).


In [ ]:
# 1. Drop duplicates
df_clean = df.drop_duplicates().copy()

# 2. Fix Data Types (e.g., TotalCharges might be read as object if there are spaces)
if df_clean['TotalCharges'].dtype == 'object':
    # Coerce errors to NaN so we can impute or drop them
    df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# 3. Handle Missing Values
# Impute missing numerical values with median
for col in df_clean.select_dtypes(include=['float64', 'int64']).columns:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"Imputed missing values in {col} with median: {median_val}")

# Impute missing categorical values with mode
for col in df_clean.select_dtypes(include=['object']).columns:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"Imputed missing values in {col} with mode: {mode_val}")

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())


### Outlier Detection & Handling
We will use boxplots to visualize outliers for continuous variables like `MonthlyCharges` and `TotalCharges`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df_clean['MonthlyCharges'], ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot of Monthly Charges')

sns.boxplot(y=df_clean['TotalCharges'], ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot of Total Charges')

plt.tight_layout()
plt.show()


*Observation:* We will retain these values for now as high charges are often valid in subscription models, representing power users or premium tiers.


## 4. Exploratory Data Analysis (EDA)
Let's inspect meaningful distributions and relationships, particularly concerning Customer Churn.


In [ ]:
# Churn Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='Churn', palette='Set2')
plt.title('Customer Churn Distribution')
plt.show()

churn_rate = df_clean['Churn'].value_counts(normalize=True) * 100
print(f"Churn Rate:\n{churn_rate}")


In [ ]:
# Churn by Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x='ContractType', hue='Churn', palette='Set1')
plt.title('Churn by Contract Type')
plt.show()


*Insight:* Month-to-month contracts typically have a significantly higher churn rate compared to one or two-year contracts.


In [ ]:
# Distribution of Monthly Charges by Churn
plt.figure(figsize=(8, 5))
sns.histplot(data=df_clean, x='MonthlyCharges', hue='Churn', kde=True, bins=30, palette='viridis')
plt.title('Distribution of Monthly Charges by Churn')
plt.show()


## 5. Export Cleaned Dataset
Finally, we export the cleaned data to a new CSV file.


In [ ]:
# Export clean dataset
df_clean.to_csv('customer_churn_cleaned.csv', index=False)
print("Clean dataset saved as 'customer_churn_cleaned.csv'.")


## Summary & Conclusion

### Data Quality Decisions:
1. **Duplicates:** Removed to ensure accuracy.
2. **Missing Values:** Numerical columns imputed using median to be robust against outliers; categorical columns imputed using mode.
3. **Data Types:** Coerced `TotalCharges` to numeric, addressing any potential hidden string values.
4. **Outliers:** Retained, as premium subscriptions can naturally result in higher `TotalCharges`.

### Initial Patterns Found:
- **Contract Type:** Customers on Month-to-Month contracts exhibit the highest churn, suggesting a lack of long-term commitment.
- **Monthly Charges:** Higher churn is often observed in certain higher charge brackets, which may indicate price sensitivity.
